Imports

In [1]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-27 10:16:46.329039: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-27 10:16:46.333736: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-27 10:16:46.333749: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [49]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=1:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)
cluster.scale(jobs=1)

In [50]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [51]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat)

In [52]:
test.extract_params(client)

In [53]:
ex=test.by_cre_parameters.result()

In [54]:
ex

In [10]:
ex['nb']

{'brain': C(cre_id)[everybody]    107.751761
 C(cre_id)[neurogene]     96.277625
 C(cre_id)[nobody]         1.023125
 C(cre_id)[redgene]       30.743570
 C(cre_id)[somebody]       9.997774
 dtype: float64,
 'blood': C(cre_id)[everybody]    103.512591
 C(cre_id)[neurogene]     15.495550
 C(cre_id)[nobody]         1.945194
 C(cre_id)[redgene]      105.600359
 C(cre_id)[somebody]      12.489160
 dtype: float64}

In [11]:
#store in df with uniq_predictor
test.by_cre.result().uniq_predictor

{'brain': (      C(cre_id)[everybody]  C(cre_id)[neurogene]  C(cre_id)[nobody]  \
  0                        0                     0                  1   
  440                      0                     0                  0   
  904                      1                     0                  0   
  1355                     0                     0                  0   
  1798                     0                     1                  0   
  
        C(cre_id)[redgene]  C(cre_id)[somebody]  
  0                      0                    0  
  440                    0                    1  
  904                    0                    0  
  1355                   1                    0  
  1798                   0                    0  ,
        C(rep_id)[1]  C(rep_id)[2]  C(rep_id)[3]
  0                1             0             0
  4761             0             1             0
  9536             0             0             1),
 'blood': (      C(cre_id)[everybody]  C(cre_id)[ne

In [12]:
assert test.by_cre.result().uniq_predictor.keys() == ex['nb'].keys()

ret={}

key='brain'

ex['nb'][key]

X,Z= test.by_cre.result().uniq_predictor[key]



#for key in ex['nb'].keys():
    #


In [13]:
X=X.sort_values(X.columns.to_list(),ascending=False)
X

,C(cre_id)[everybody],C(cre_id)[neurogene],C(cre_id)[nobody],C(cre_id)[redgene],C(cre_id)[somebody]
904,1,0,0,0,0
1798,0,1,0,0,0
0,0,0,1,0,0
1355,0,0,0,1,0
440,0,0,0,0,1


In [14]:
ex['nb'][key]

C(cre_id)[everybody]    107.751761
C(cre_id)[neurogene]     96.277625
C(cre_id)[nobody]         1.023125
C(cre_id)[redgene]       30.743570
C(cre_id)[somebody]       9.997774
dtype: float64

In [15]:
X['nb']=ex['nb'][key].to_numpy()

In [16]:
X

,C(cre_id)[everybody],C(cre_id)[neurogene],C(cre_id)[nobody],C(cre_id)[redgene],C(cre_id)[somebody],nb
904,1,0,0,0,0,107.751761
1798,0,1,0,0,0,96.277625
0,0,0,1,0,0,1.023125
1355,0,0,0,1,0,30.743570
440,0,0,0,0,1,9.997774


In [48]:
cluster.close()